Setup and Load Data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load the raw data (semicolon-separated KoboToolbox export)
file_path = "Zim_cooking_fuels_-_all_versions_-_False_-_2026-11-12-06-31-58.csv"
df = pd.read_csv(file_path, sep=';')

print(f"Loaded {len(df)} records.")
print(df['domain'].value_counts())

Loaded 34 records.
domain
urban    18
rural    16
Name: count, dtype: int64


Defining the Indicator Logic

In [2]:
# Clean fuels: electricity, gas, solar
clean_fuels = ['electricity', 'gas', 'solar']
df['is_clean_fuel'] = df['primary_fuel'].isin(clean_fuels)

# Solid fuels (eligible for the improved-stove question): firewood, charcoal, dung/crop residue
# Note: paraffin is neither clean nor solid in this analysis.
solid_fuels = ['firewood', 'charcoal', 'dung_or_crop_residue']
df['is_solid_fuel'] = df['primary_fuel'].isin(solid_fuels)

# Convert expenditure to numeric; blanks / non-numeric become NaN
df['fuel_expenditure_usd_clean'] = pd.to_numeric(df['fuel_expenditure_usd'], errors='coerce')

print("Flags created.")

Flags created.


 Indicator 1 - Clean Fuel % (Overall and by Domain)

In [3]:
# Denominator = valid answers to primary_fuel (exclude 'dont_know' and 'refused')
valid_fuel_df = df[~df['primary_fuel'].isin(['dont_know', 'refused'])]

clean_fuel_stats = valid_fuel_df.groupby('domain')['is_clean_fuel'].agg(['count', 'sum'])
clean_fuel_stats.columns = ['Valid_Denominator', 'Clean_Fuel_Numerator']
clean_fuel_stats['Percentage'] = (clean_fuel_stats['Clean_Fuel_Numerator'] / clean_fuel_stats['Valid_Denominator'] * 100).round(1)

total_valid = clean_fuel_stats['Valid_Denominator'].sum()
total_clean = clean_fuel_stats['Clean_Fuel_Numerator'].sum()
overall_pct = round((total_clean / total_valid) * 100, 1)

print("--- INDICATOR 1: CLEAN FUEL % ---")
print(clean_fuel_stats)
print(f"\nOverall: {total_clean}/{total_valid} = {overall_pct}%")

# Audit: raw counts of primary_fuel by domain
print("\n--- AUDIT: primary_fuel by domain ---")
print(pd.crosstab(df['primary_fuel'], df['domain'], margins=True))

--- INDICATOR 1: CLEAN FUEL % ---
        Valid_Denominator  Clean_Fuel_Numerator  Percentage
domain                                                     
rural                  16                     4        25.0
urban                  17                    17       100.0

Overall: 21/33 = 63.6%

--- AUDIT: primary_fuel by domain ---
domain                rural  urban  All
primary_fuel                           
charcoal                  2      0    2
dung_or_crop_residue      4      0    4
electricity               0      8    8
firewood                  3      0    3
gas                       2      5    7
paraffin                  3      0    3
refused                   0      1    1
solar                     2      4    6
All                      16     18   34


**Analysis of Indicator 1: Clean Fuel Use**
*   **Overall:** 63.6% of valid respondents use clean fuel (21 out of 33).
*   **Urban:** 100.0% (17/17) of urban households use clean fuels.
*   **Rural:** Only 25.0% (4/16) of rural households use clean fuels.

**Conclusion:** There is a stark contrast between urban and rural areas. Every valid urban household in our pilot used electricity, gas, or solar. In rural areas, 9 of 16 households (56%) mainly use solid fuels (firewood, charcoal, dung/crop residue), 3 use paraffin and only 4 use a clean fuel (2 gas, 2 solar). The pilot is consistent with the hypothesis that urban households have moved to clean fuels while rural households lag behind, but with n = 33 this is only indicative and the reasons (e.g. cost or infrastructure) cannot be tested with these data.

**Note on Denominator:** The valid denominator is 33 (not 34) because one urban respondent (ZIM-0009) refused to answer the primary fuel question. This refusal was excluded from the calculation. The urban denominator is therefore 17 (of 18 interviewed) and the rural denominator is 16.

Visualisation 1 - Bar Chart for Clean Fuel

In [4]:
# Create the bar chart
plt.figure(figsize=(8, 5))
order = clean_fuel_stats.sort_index()
bars = plt.bar(order.index, order['Percentage'], color=['#2c7fb8', '#7fcdbb'])

plt.title('Clean Fuel Use by Domain (Pilot Data)', fontsize=14)
plt.ylabel('Percentage of Households (%)', fontsize=12)
plt.xlabel('Domain', fontsize=12)
plt.ylim(0, 110)

for bar, num, den in zip(bars, order['Clean_Fuel_Numerator'], order['Valid_Denominator']):
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 2, f'{yval}%\n({num}/{den})',
             ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.show()

Indicator 2 - Improved Stove % (The Eligible Subgroup)

In [5]:
# Eligible subgroup: solid-fuel users with a valid 'yes' or 'no' stove answer
stove_df = df[df['is_solid_fuel']]
valid_stove_df = stove_df[stove_df['has_improved_stove'].isin(['yes', 'no'])]

stove_counts = valid_stove_df.groupby('domain')['has_improved_stove'].count()
stove_yes = valid_stove_df.groupby('domain')['has_improved_stove'].apply(lambda x: (x == 'yes').sum())

print("--- INDICATOR 2: IMPROVED STOVE % (Among Solid Fuel Users) ---")
for d in ['urban', 'rural']:
    n = int(stove_counts.get(d, 0))
    y = int(stove_yes.get(d, 0))
    if n == 0:
        print(f"{d.capitalize()}: N/A (n=0)")
    else:
        print(f"{d.capitalize()}: {y}/{n} = {round(y/n*100, 1)}%")

# Audit: solid-fuel users and why some are excluded from the denominator
print("\nSolid-fuel users by primary_fuel and stove answer:")
print(pd.crosstab(stove_df['primary_fuel'], stove_df['has_improved_stove'].fillna('blank (not asked)')))

--- INDICATOR 2: IMPROVED STOVE % (Among Solid Fuel Users) ---
Urban: N/A (n=0)
Rural: 2/4 = 50.0%

Solid-fuel users by primary_fuel and stove answer:
has_improved_stove    blank (not asked)  no  refused  yes
primary_fuel                                             
charcoal                              0   0        0    2
dung_or_crop_residue                  4   0        0    0
firewood                              0   2        1    0


**Analysis of Indicator 2: Improved Stove Ownership (Eligible Subgroup)**
*   **Urban:** N/A (n=0). There are no eligible urban households to calculate this indicator.
*   **Rural:** 50.0% (2/4). Half of the rural solid-fuel users with a valid answer reported owning an improved cookstove.

**Conclusion on the Eligible Subgroup:** This indicator is only defined for households that use solid fuels (firewood, charcoal, or dung/crop residue). All 17 valid urban respondents used clean fuels, so none were eligible. The urban denominator is zero and the rate is "Not Applicable" (N/A), not 0%. We cannot estimate urban stove ownership because there are no urban solid-fuel users in our sample.

**Denominator note:** 9 rural households use solid fuels, but only 4 are in the denominator: the 4 dung/crop-residue users have no stove answer recorded (blank) and 1 firewood user (ZIM-0020) answered "refused". Only the 4 households that answered yes/no are counted.

**Conclusion on Rural Stoves:** With a denominator of only n = 4, this estimate is highly uncertain and is only a small snapshot of the pilot, not the true rural population.

Indicator 3 - Mean Expenditure by Domain

In [6]:
# Mean, count and standard deviation of monthly fuel expenditure by domain
exp_stats = df.groupby('domain')['fuel_expenditure_usd_clean'].agg(['mean', 'count', 'std']).round(2)

overall_mean = round(df['fuel_expenditure_usd_clean'].mean(), 2)
overall_n = df['fuel_expenditure_usd_clean'].count()

print("--- INDICATOR 3: MEAN MONTHLY FUEL EXPENDITURE ---")
print(exp_stats)
print(f"\nOverall mean: ${overall_mean} (n={overall_n})")

# Audit: why are responses missing?
print("\nMissing expenditure by fuel_purchased answer:")
print(df[df['fuel_expenditure_usd_clean'].isna()][['case_id', 'domain', 'fuel_purchased']].to_string(index=False))

--- INDICATOR 3: MEAN MONTHLY FUEL EXPENDITURE ---
         mean  count    std
domain                     
rural    8.25     12   9.39
urban   32.06     16  28.37

Overall mean: $21.86 (n=28)

Missing expenditure by fuel_purchased answer:
 case_id domain fuel_purchased
ZIM-0033  rural      dont_know
ZIM-0009  urban        refused
ZIM-0014  urban      dont_know
ZIM-0027  rural      dont_know
ZIM-0034  rural      dont_know
ZIM-0026  rural      dont_know


**Analysis of Indicator 3: Mean Monthly Fuel Expenditure**
*   **Overall Mean:** $21.86 USD (based on 28 valid responses).
*   **Urban Mean:** $32.06 USD (n=16), with a high standard deviation of 28.37.
*   **Rural Mean:** $8.25 USD (n=12), with a standard deviation of 9.39.

**Conclusion:** Urban households spend about four times as much on cooking fuel per month as rural households ($32.06 vs $8.25). Urban households mostly buy electricity or gas, whereas many rural households use firewood or crop residue that they do not buy ($0 spent).

**Note on Variance:** The urban standard deviation ($28.37) is large relative to its mean, so spending varies widely between urban households (from $0 for solar users to $80 for electricity users). The rural standard deviation is smaller in absolute terms, but it is also large relative to its mean ($8.25), because half of the rural respondents report $0 and the rest spend $10 to $25. With n = 16 and n = 12, the means should be treated as indicative only.

**Note on Missing Data:** The denominator dropped from 34 records to 28 valid responses (Urban n=16, Rural n=12). The 6 missing values are 1 "refused" (ZIM-0009) and 5 "don't know" (ZIM-0014, ZIM-0026, ZIM-0027, ZIM-0033, ZIM-0034). Households that do not buy fuel and reported $0 (9 households) are valid zeros and ARE included in the means.

Visualisation 2 - Boxplot for Expenditure

In [7]:
# Boxplot of expenditure by domain
fig, ax = plt.subplots(figsize=(8, 5))
df.boxplot(column='fuel_expenditure_usd_clean', by='domain', grid=False, patch_artist=True,
           boxprops=dict(facecolor='lightblue'), ax=ax)
ax.set_title('Distribution of Monthly Fuel Expenditure (USD)', fontsize=14)
fig.suptitle('')  # remove the automatic pandas subtitle
ax.set_ylabel('Expenditure (USD)', fontsize=12)
ax.set_xlabel('Domain', fontsize=12)
plt.show()

Data Quality Check

In [8]:
print("--- DATA QUALITY: MISSING / INAPPLICABLE ---")
# Clean-fuel users should NOT have a stove answer (routing: stove question is for solid-fuel users only)
clean_but_stove_answered = df[(df['is_clean_fuel']) & (df['has_improved_stove'].notna())]
print(f"Clean fuel users who had stove data (should be 0): {len(clean_but_stove_answered)}")

# Zero expenditures are valid (household does not buy fuel)
zero_exp = df[df['fuel_expenditure_usd_clean'] == 0]
print(f"Households with $0 expenditure (valid zeros): {len(zero_exp)}")

# Completeness
print(f"Records with Record_Interview_outcome = partial: {(df['Record_Interview_outcome'] == 'partial').sum()}")
print(f"Duplicate case_id values: {df['case_id'].duplicated().sum()}")

# Final audit of the raw primary_fuel counts
print("\n--- FINAL AUDIT: RAW FUEL COUNTS ---")
print(df['primary_fuel'].value_counts(dropna=False))

--- DATA QUALITY: MISSING / INAPPLICABLE ---
Clean fuel users who had stove data (should be 0): 0
Households with $0 expenditure (valid zeros): 9
Records with Record_Interview_outcome = partial: 1
Duplicate case_id values: 0

--- FINAL AUDIT: RAW FUEL COUNTS ---
primary_fuel
electricity             8
gas                     7
solar                   6
dung_or_crop_residue    4
firewood                3
paraffin                3
charcoal                2
refused                 1
Name: count, dtype: int64
